# Distributional Semantics & SAT Analogies
**Drop this notebook in the folder that contains** `dist_sim_data.txt`, `SAT-package-V3.txt`, and your embedding files.

Run cells top-to-bottom. Edit the `DATA_DIR` if needed.

In this part, I built a small distributional semantic model from the toy corpus. First, I cleaned and tokenized the text into lowercase words and created a vocabulary. Then I made a co-occurrence matrix (C) where each entry shows how often two words appear next to each other. I counted both (w, c) and (c, w) pairs to make the matrix symmetric. Next, I multiplied all counts by 10 to make the data less sparse and added 1 to every cell for smoothing. This ensures there are no zero values. Then I calculated Positive Pointwise Mutual Information (PPMI) using the formula in the pdf. 
PPMI shows how strongly two words co-occur compared to chance. Negative values are set to zero. This highlights meaningful word relationships and downplays random ones. Finally, I used Singular Value Decomposition (SVD) to reduce dimensions and create 3-D word vectors. I verified that multiplying U * E * Vt reconstructs the original matrix. The reduced vectors still capture important semantic information but in a smaller, more compact space.

In [ ]:
#   I did this for no reason :), just make sure the files are in the same folder as this notebook txt file 
from pathlib import Path
import math, re, itertools, gzip, io
import numpy as np
import pandas as pd

DATA_DIR = Path('.') 
toy_corpus_path = DATA_DIR/'dist_sim_data.txt'
sat_path = DATA_DIR/'SAT-package-V3.txt'
google_path = DATA_DIR/'GoogleNews-vectors-rcv_vocab.txt'  
print('Found:', toy_corpus_path.exists(), sat_path.exists(), google_path.exists())


Found: True True True


## Part 1

In [ ]:
def read_toy_corpus(path: Path):
    text = Path(path).read_text(encoding='utf-8').strip().lower()
 
    toks = [t for t in re.split(r'[^a-z]+', text) if t]
    return toks

def build_vocab(tokens):
   
    idx = {}
    for t in tokens:
        if t not in idx: idx[t] = len(idx)
    vocab = list(idx.keys())
    word2id = idx
    id2word = {i:w for w,i in word2id.items()}
    return vocab, word2id, id2word

def cooc_adjacent(tokens, word2id):
    n = len(word2id)
    C = np.zeros((n,n), dtype=np.float64)
    for a,b in zip(tokens, tokens[1:]):
        i = word2id[a]; j = word2id[b]
        C[i,j] += 1.0
        C[j,i] += 1.0  # count (c,w) 
    # scale ×10 and add +1 smoothing
    C = C * 10.0
    C += 1.0
    return C

def ppmi(C):
    total = C.sum()
    pw = C.sum(axis=1, keepdims=True) / total
    pc = C.sum(axis=0, keepdims=True) / total
    pwc = C / total
    
    denom = pw @ pc
    with np.errstate(divide='ignore', invalid='ignore'):
        pmi = np.log(np.where(denom>0, pwc/denom, 1.0))
    pmi[np.isneginf(pmi)] = 0.0
    P = np.maximum(pmi, 0.0)  
    return P

def euclid(a,b):
    return float(np.linalg.norm(a-b))

def svd_reduce(P, k=3):
   
    U,S,Vt = np.linalg.svd(P, full_matrices=False)
    V = Vt.T
    Vk = V[:, :k]

    Xk = P @ Vk
    return Xk, (U,S,Vt)

def distance_table(words, vectors, word2id):
    rows = []
    for a,b in words:
        if a not in word2id or b not in word2id:
            d = None
        else:
            ia, ib = word2id[a], word2id[b]
            d = euclid(vectors[ia], vectors[ib])
        rows.append({'w1':a, 'w2':b, 'euclid':d})
    return pd.DataFrame(rows)


tokens = read_toy_corpus(toy_corpus_path)
vocab, word2id, id2word = build_vocab(tokens)
C = cooc_adjacent(tokens, word2id)
P = ppmi(C)
X3, svd_bits = svd_reduce(P, k=3)

print('Vocab size:', len(vocab))


Vocab size: 7


In [ ]:

target = 'dogs'
if target in word2id:
    i = word2id[target]
    before = C[i]
    after  = P[i]

 
    top_before = np.argsort(-before)[:10]
    top_after  = np.argsort(-after)[:10]

    print(f"Top contexts for '{target}' BEFORE PPMI:")
    for j in top_before:
        print(id2word[j], before[j])

    print(f"\nTop contexts for '{target}' AFTER PPMI:")
    for j in top_after:
        print(id2word[j], after[j])
else:
    print(f"'dogs' not in corpus vocab; pick another token from: {vocab[:20]}")


Top contexts for 'dogs' BEFORE PPMI:
the 131.0
bite 31.0
like 11.0
men 1.0
dogs 1.0
feed 1.0
women 1.0

Top contexts for 'dogs' AFTER PPMI:
bite 1.082232416670013
the 0.6482019596813942
feed 0.0
men 0.0
dogs 0.0
women 0.0
like 0.0


In [ ]:

pairs = [('women','men'), ('women','dogs'), ('men','dogs'),
         ('feed','like'), ('feed','bite'), ('like','bite')]

tab_full = distance_table(pairs, P, word2id)
tab_3d = distance_table(pairs, X3, word2id)

tab_full, tab_3d


(      w1    w2    euclid
 0  women   men  0.496481
 1  women  dogs  1.462951
 2    men  dogs  1.285032
 3   feed  like  0.646627
 4   feed  bite  1.511743
 5   like  bite  1.222483,
       w1    w2    euclid
 0  women   men  0.190256
 1  women  dogs  1.030950
 2    men  dogs  0.887532
 3   feed  like  0.458631
 4   feed  bite  1.078175
 5   like  bite  0.850333)

## Part 2 — SAT Analogies (relation vectors)

In [ ]:

def load_txt_embeddings(path: Path, limit=None):

    w2v = {}
    if not path.exists():
        return w2v
    with path.open('r', encoding='utf-8', errors='ignore') as f:
        for line_i, line in enumerate(f):
            parts = line.rstrip().split()
            if len(parts) < 3: 
                continue
            w = parts[0]
            try:
                vec = np.array([float(x) for x in parts[1:]], dtype=np.float32)
            except ValueError:
                continue
            w2v[w] = vec
            if limit and len(w2v) >= limit:
                break
    return w2v


google = load_txt_embeddings(google_path)
print('Loaded Google vectors:', len(google))




Loaded Google vectors: 140922


In [ ]:

import re, itertools

def iter_sat_questions(path: Path):
    lines = [l.strip() for l in path.read_text(encoding='utf-8', errors='ignore').splitlines()]
    i = 0
    n = len(lines)
    while i < n:
        
        if not lines[i] or lines[i].startswith('#'):
            i += 1
            continue

      
        if not re.match(r'^\d+', lines[i]):
            i += 1
            continue

        
        if i + 7 >= n:
            break
        pairs = []
        for j in range(i + 1, i + 7):
            parts = lines[j].split()
            if len(parts) < 2:
                break
            w1, w2 = parts[0].lower(), parts[1].lower()
            pairs.append((w1, w2))
        if len(pairs) != 6:
            i += 1
            continue

    
        gold_line = lines[i + 7].strip().lower()
        if gold_line not in ['a', 'b', 'c', 'd', 'e']:
            i += 1
            continue
        gold = 'abcde'.index(gold_line)

        prompt = pairs[0]
        options = pairs[1:]
        yield (prompt[0], prompt[1], options, gold)

        i += 8  # skip to next block

# ---- Smoke test ----
sample = list(itertools.islice(iter_sat_questions(sat_path), 3))
print("Parsed problems:", len(sample))
for (A, B, opts, gold) in sample:
    print(f"{A}:{B} -> gold={gold}, options={opts}")


Parsed problems: 3
lull:trust -> gold=2, options=[('balk', 'fortitude'), ('betray', 'loyalty'), ('cajole', 'compliance'), ('hinder', 'destination'), ('soothe', 'passion')]
ostrich:bird -> gold=0, options=[('lion', 'cat'), ('goose', 'flock'), ('ewe', 'sheep'), ('cub', 'bear'), ('primate', 'monkey')]
word:language -> gold=2, options=[('paint', 'portrait'), ('poetry', 'rhythm'), ('note', 'music'), ('tale', 'story'), ('week', 'year')]


In [ ]:
def relation_score(w2v, pair_prompt, pair_option):
    (A, B) = pair_prompt
    (X, Y) = pair_option
    if A not in w2v or B not in w2v or X not in w2v or Y not in w2v:
        return None
    rel = w2v[B] - w2v[A]
    rel_opt = w2v[Y] - w2v[X]
    #Euclidean distance between relation vectors
    return float(np.linalg.norm(rel - rel_opt))


def evaluate_sat(w2v, path: Path, max_q=None):
    right = 0
    total = 0
    covered = 0
    for qi, (A, B, cands, gold) in enumerate(iter_sat_questions(path)):
        if max_q and qi >= max_q:
            break
        scores = [relation_score(w2v, (A, B), c) for c in cands]
        total += 1
        if any(s is None for s in scores):
            continue
        covered += 1
        pred = int(np.argmin(scores))
        if pred == gold:
            right += 1
    acc = (right / covered * 100.0) if covered else 0.0
    coverage = (covered / total * 100.0) if total else 0.0
    return {'acc_percent': acc, 'coverage_percent': coverage,
            'covered': covered, 'total': total}


google_res = evaluate_sat(google, sat_path)

pd.DataFrame([{'model': 'GoogleNews', **google_res}])


,model,acc_percent,coverage_percent,covered,total
0,GoogleNews,37.40458,68.947368,131,190
